In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import torch.backends.cudnn as cudnn

import torchvision
from torchvision import datasets, models, transforms

import numpy as np
import matplotlib.pyplot as plt
import time
import os
import copy

In [2]:
# Definir transformaciones de datos para entrenamiento, validación y prueba
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

In [17]:
data_dir = os.path.join('..', '..', 'data')

# Cargar dataset
image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x]) for x in ['train', 'val', 'test']}

# Se usa num_workers=0 para Windows para prevenir errores de multiprocesamiento en Jupyter
dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=4, shuffle=True, num_workers=0) for x in ['train', 'val', 'test']}
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val', 'test']}
class_names = image_datasets['train'].classes
print(f"Clases cargadas: {class_names}")
print(f"Tamaños del dataset: {dataset_sizes}")

Clases cargadas: ['gray_leaf_spot', 'healthy', 'magnesium_deficiency']
Tamaños del dataset: {'train': 1260, 'val': 270, 'test': 270}


In [18]:
# ---------------------------------
# Arquitectura de la red neuronal
# ---------------------------------

# Descargar ResNet-18 preentrenada en ImageNet
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Congelamos las capas internas para evitar que se actualicen durante el entrenamiento
for param in model.parameters():
    param.requires_grad = False

# Cambiamos la última capa (fully connected) para que tenga el número de clases de nuestro dataset
num_ftrs = model.fc.in_features

# len(class_names) es el número de clases en nuestro dataset
model.fc = nn.Linear(num_ftrs, len(class_names))

# Mover el modelo a la GPU si está disponible
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\usuario/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100.0%


In [ ]:
# Función de pérdida estándar para clasificación: CrossEntropyLoss
criterion = nn.CrossEntropyLoss()

# Optimizador: solo actualizamos los parámetros de la última capa (model.fc)
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

In [ ]:
num_epochs = 15

for epoch in range(num_epochs):
    print(f'Epoch {epoch+1}/{num_epochs}')
    print('-' * 10)
    
    # Cada epoch tiene una fase de entrenamiento y una de validación
    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()  # Activa el modo de entrenamiento (aplica Dropout, etc.)
        else:
            model.eval()   # Activa el modo de evaluación (congela el modelo)
            
        running_loss = 0.0
        running_corrects = 0
        
        # Iterar sobre los lotes (batches) de imágenes del DataLoader
        for inputs, labels in dataloaders[phase]:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            # Limpiar los gradientes del optimizador
            optimizer.zero_grad()
            
            # Forward: El modelo intenta adivinar qué enfermedad es
            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)
                
                # Backward + Optimizar solo si estamos en fase de entrenamiento
                if phase == 'train':
                    loss.backward()  # Calcula el error
                    optimizer.step() # Ajusta los pesos matemáticos
            
            # Estadísticas
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)
            
        epoch_loss = running_loss / dataset_sizes[phase]
        epoch_acc = running_corrects.double() / dataset_sizes[phase]
        
        print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

Epoch 1/10
----------


UnidentifiedImageError: cannot identify image file <_io.BufferedReader name='..\\..\\data\\train\\healthy\\healthy_maize_desease_v1.1_real_2277.jpg'>